# Efficiency optimization

Checking every combination of parameters to optimize for the most efficient setup
1. virtualisation strategy: none vs chunk vs cluster
2. thinking: on vs off
3. group summary of virtual folders
4. max_branches
5. decision_cap

Queries tested:
- What are the Ten Steps To CAP Laboratory Accreditation?
- What is the minimum concentration of cfDNA aliquot needed for pWGS?
- What reagents are used with the Illumina NextSeq 2000?

In [1]:
%pip install ollama numpy pandas matplotlib tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, re, json, time, hashlib, itertools, statistics
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional
import ollama
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

OLLAMA_URL  = "http://localhost:11528"        # tunnelled endpoint, same as prototype7
AGENT_MODEL = "gpt-oss:120b"
EMBED_MODEL = "nomic-embed-text"              # only the cluster strategy touches this
CACHE_DIR   = Path("tree_cache")
TREE_FILE   = CACHE_DIR / "corpus_tree.json"
OUT_DIR     = Path("speedtest_out")
OUT_DIR.mkdir(exist_ok=True)

# held constant so the race is fair
VIRTUAL_TARGET    = 6
MAX_STEPS         = 40
MEMORY_MAX_ITEMS  = 20
MAX_EVIDENCE      = 12
SYNTH_NUM_PREDICT = 1000000                   # final answer left uncapped
GROUP_NUM_PREDICT = 400
KEEP_ALIVE        = "30m"                      # keep models warm between configs

# placeholders, the driver overwrites these per config
VIRTUAL_STRATEGY     = "cluster"
VIRTUAL_SUMMARY_LLM  = True
MAX_BRANCH           = 8
DECISION_NUM_PREDICT = 256

# same queries for every config
QUERIES = [
    "What are the Ten Steps To CAP Laboratory Accreditation?",
    "What is the minimum concentration of cfDNA aliquot needed for pWGS?",
    "What reagents are used with the Illumina NextSeq 2000?",
]
REPEATS = 1                                   # bump if timings feel noisy

GRID = {
    "model":         [AGENT_MODEL],           # add gpt-oss:20b etc if you have it pulled
    "strategy":      ["none", "chunk", "cluster"],
    "thinking":      [False, True],           # reasoning on is the big latency lever
    "group_summary": ["heuristic", "llm"],    # heuristic is free, llm costs a call
    "max_branch":    [5, 8, 12],
    "decision_cap":  [256],
}
print("queries:", len(QUERIES), "| repeats:", REPEATS)
print("grid axes:", {k: len(v) for k, v in GRID.items()})

queries: 3 | repeats: 1
grid axes: {'model': 1, 'strategy': 3, 'thinking': 2, 'group_summary': 2, 'max_branch': 3, 'decision_cap': 1}


/opt/homebrew/Cellar/jupyterlab/4.5.7_1/libexec/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
client = ollama.Client(host=OLLAMA_URL, timeout=600)
_THINK = {"use": True}                         # driver flips this per config
_stats = {}


def _reset_stats():
    # fresh counters per config
    _stats.clear()
    for k in ("chat_time", "embed_time", "decision_calls", "synth_calls", "group_calls", "embed_calls"):
        _stats[k] = 0.0 if "time" in k else 0


def _model_names(r):
    # dig names out, ollama versions disagree on shape
    raw = r.get("models", []) if hasattr(r, "get") else getattr(r, "models", [])
    out = []
    for m in raw:
        n = getattr(m, "model", None) or getattr(m, "name", None)
        if n is None and isinstance(m, dict):
            n = m.get("model") or m.get("name")
        if n:
            out.append(n)
    return out


def _chat_call(prompt, num_predict):
    # the only real model call, timed and retried a few times
    opts = {"temperature": 0, "num_predict": num_predict}
    last = None
    for attempt in range(3):
        kw = dict(model=AGENT_MODEL, messages=[{"role": "user", "content": prompt}], options=opts, keep_alive=KEEP_ALIVE)
        if _THINK["use"]:
            kw["think"] = True
        t = time.perf_counter()
        try:
            r = client.chat(**kw)
            _stats["chat_time"] += time.perf_counter() - t
            return r["message"]["content"].strip()
        except TypeError:
            _THINK["use"] = False              # build cant think, drop it
        except Exception as e:
            last = e
            time.sleep(2)
    raise RuntimeError("chat failed after retries: " + str(last))


def _chat_text(prompt):
    # decisions and the final answer both come here, split by a marker phrase
    is_synth = "Write a clear, specific answer" in prompt
    npred = SYNTH_NUM_PREDICT if is_synth else DECISION_NUM_PREDICT
    out = _chat_call(prompt, npred)
    _stats["synth_calls" if is_synth else "decision_calls"] += 1
    return out


def _combine_text(summaries, label):
    # short blurb for a virtual group, only when group_summary is llm
    joined = "\n".join("- " + s for s in summaries)
    prompt = "In 2-4 sentences, describe what this group called '" + label + "' covers so an agent can decide whether to explore it. Respond with ONLY the description.\n\n" + joined
    out = _chat_call(prompt, GROUP_NUM_PREDICT)
    _stats["group_calls"] += 1
    return out


def _embed(text):
    # one summary to a vector for clustering, None on failure so it falls back to chunk
    t = time.perf_counter()
    try:
        v = client.embeddings(model=EMBED_MODEL, prompt=text or " ")["embedding"]
        _stats["embed_time"] += time.perf_counter() - t
        _stats["embed_calls"] += 1
        return v
    except Exception:
        return None


AVAILABLE = _model_names(client.list())        # what you actually have pulled
print("models available:", ", ".join(AVAILABLE) or "none")

models available: nomic-embed-text:latest, gpt-oss:120b, gemma3:27b, gemma3:270m, llama3.1:latest, llama3.1:8b, medgemma:27b, gpt-oss:20b, codellama:latest, llama3.1:70b, mistral:latest, llama3:70b-instruct, mistral-small3.1:latest


**Cells below are the prototype 7 agent — virtualisation + MemWalker, carried over unchanged. This is the system under test; the benchmark drives it via the instrumented `_chat_text` / `_combine_text` / `_embed` defined above.**

In [4]:
import math


@dataclass
class TreeNode:
    node_id:   str
    node_type: str
    name:      str
    path:      str
    summary:   str
    content:   str = ""
    children:  List["TreeNode"] = field(default_factory=list)
    metadata:  Dict[str, Any]   = field(default_factory=dict)

    @classmethod
    def from_dict(cls, d: Dict) -> "TreeNode":
        node = cls(node_id=d["node_id"], node_type=d["node_type"], name=d["name"],
                   path=d.get("path", ""), summary=d.get("summary", ""),
                   content=d.get("content", ""), metadata=d.get("metadata", {}))
        node.children = [cls.from_dict(c) for c in d.get("children", [])]
        return node

    def is_leaf(self) -> bool:
        return self.node_type == "chunk"

    def count_leaves(self) -> int:
        return 1 if self.is_leaf() else sum(c.count_leaves() for c in self.children)


def _vid(parent_id: str, tag: str) -> str:
    return "v" + hashlib.md5(f"{parent_id}|{tag}".encode()).hexdigest()[:11]


def _clip(text: str, n: int) -> str:
    t = re.sub(r"\s+", " ", text or "").strip()
    return t if len(t) <= n else t[:n] + " …"


_embed_cache: Dict[str, Any] = {}


def _embed_node(node: "TreeNode"):
    if node.node_id in _embed_cache:
        return _embed_cache[node.node_id]
    vec = _embed((node.name + ". " + (node.summary or ""))[:2000])
    _embed_cache[node.node_id] = vec
    return vec


def _kmeans(vectors, k: int, iters: int = 25, seed: int = 0):
    X = np.asarray(vectors, dtype=float)
    n = len(X)
    k = max(1, min(k, n))
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    X = X / np.clip(norms, 1e-9, None)          # cosine-ish via unit vectors
    rng = np.random.default_rng(seed)
    C = X[rng.choice(n, size=k, replace=False)].copy()
    labels = np.full(n, -1)
    for _ in range(iters):
        d = ((X[:, None, :] - C[None, :, :]) ** 2).sum(-1)
        new = d.argmin(1)
        if np.array_equal(new, labels):
            break
        labels = new
        for j in range(k):
            pts = X[labels == j]
            C[j] = pts.mean(0) if len(pts) else X[rng.integers(n)]
    return labels.tolist()


def _group_summary(members: List["TreeNode"], label: str) -> str:
    if VIRTUAL_SUMMARY_LLM:
        try:
            return _combine_text([m.summary for m in members], label=label)
        except Exception:
            pass
    lines = [f"- {m.name}: {_clip(m.summary, 160)}" for m in members[:MAX_BRANCH]]
    more = f"\n- …and {len(members) - MAX_BRANCH} more" if len(members) > MAX_BRANCH else ""
    return f"A group of {len(members)} related items:\n" + "\n".join(lines) + more


def _make_vgroup(parent: "TreeNode", members: List["TreeNode"], idx: int, strategy: str) -> "TreeNode":
    vid = _vid(parent.node_id, f"{strategy}:{idx}")
    return TreeNode(node_id=vid, node_type="vgroup",
                    name=f"[group {idx + 1} · {len(members)} items]", path="",
                    summary=_group_summary(members, label=f"{parent.name} group {idx + 1}"),
                    children=list(members),
                    metadata={"virtual": True, "strategy": strategy, "size": len(members)})


def _chunk_groups(parent: "TreeNode", kids: List["TreeNode"], strategy: str = "chunk"):
    n = len(kids)
    size = math.ceil(n / MAX_BRANCH)                 # -> at most MAX_BRANCH groups
    groups = [kids[i:i + size] for i in range(0, n, size)]
    return [_make_vgroup(parent, g, i, strategy) for i, g in enumerate(groups)]


def _cluster_groups(parent: "TreeNode", kids: List["TreeNode"]):
    vecs = [_embed_node(k) for k in kids]
    if any(v is None for v in vecs):                 # no embeddings -> fall back
        return _chunk_groups(parent, kids, strategy="cluster-fallback")
    k = max(2, min(MAX_BRANCH, math.ceil(len(kids) / VIRTUAL_TARGET)))
    labels = _kmeans(vecs, k)
    buckets: Dict[int, list] = {}
    for kid, lab in zip(kids, labels):
        buckets.setdefault(lab, []).append(kid)
    # guard against a degenerate "everything in one cluster" -> no progress
    if len(buckets) < 2 or max(len(v) for v in buckets.values()) == len(kids):
        return _chunk_groups(parent, kids, strategy="cluster-fallback")
    ordered = [buckets[lab] for lab in sorted(buckets)]
    return [_make_vgroup(parent, mem, i, "cluster") for i, mem in enumerate(ordered)]


_nav_cache: Dict[tuple, List["TreeNode"]] = {}


def get_nav_children(node: "TreeNode", strategy: str) -> List["TreeNode"]:
    """Children the agent chooses among at `node` — virtualised to <= MAX_BRANCH."""
    key = (node.node_id, strategy)
    if key in _nav_cache:
        return _nav_cache[key]
    kids = node.children
    if strategy == "none" or len(kids) <= MAX_BRANCH:
        res = list(kids)
    elif strategy == "chunk":
        res = _chunk_groups(node, kids)
    elif strategy == "cluster":
        res = _cluster_groups(node, kids)
    else:
        res = list(kids)
    _nav_cache[key] = res
    return res


print("Virtual-subfolder layer ready (strategies: none | chunk | cluster)")

Virtual-subfolder layer ready (strategies: none | chunk | cluster)


In [5]:
if not TREE_FILE.exists():                              # bail early with a clear message if the tree was never built
    raise SystemExit(f"tree not found at {TREE_FILE.resolve()}; run prototype5 first")
with open(TREE_FILE, encoding="utf-8") as f:            # open the saved tree json off disk
    ROOT = TreeNode.from_dict(json.load(f))             # rebuild the whole TreeNode tree in memory so the agent can walk it

def _count(n):                                          # tiny recursive node counter, only used for the printout below
    return 1 + sum(_count(c) for c in n.children)

print("loaded tree:", ROOT.name, "|", _count(ROOT), "nodes |", ROOT.count_leaves(), "leaves |", len(ROOT.children), "top level")  # quick stats so you can see it loaded fine

loaded tree: folders | 130154 nodes | 95458 leaves | 10 top level


In [6]:
import json as _json


def _add_memory(memory: List[str], fact: str):
    fact = _clip(fact, 500)
    if fact and fact.lower() not in ("", "none", "n/a", "(empty)") and fact not in memory:
        memory.append(fact)
        del memory[:-MEMORY_MAX_ITEMS]      # keep only the most recent N


def _parse_decision(raw: str, n_options: int) -> Dict:
    s = raw.strip()
    s = re.sub(r"^```(?:json)?|```$", "", s, flags=re.M).strip()
    m = re.search(r"\{.*\}", s, flags=re.S)
    if m:
        try:
            d = _json.loads(m.group(0))
            act = str(d.get("action", "")).lower().strip()
            if act not in ("descend", "answer", "backtrack"):
                act = "descend" if n_options else "answer"
            ci = d.get("child", None)
            try:
                ci = int(ci)
            except (TypeError, ValueError):
                ci = None
            return {"action": act, "child": ci,
                    "remember": (d.get("remember") or "").strip(),
                    "reasoning": _clip(d.get("reasoning", ""), 240)}
        except Exception:
            pass
    # un-parseable -> safe default
    return {"action": "descend" if n_options else "backtrack", "child": 0 if n_options else None,
            "remember": "", "reasoning": "(unparsed model output)"}


def _ask_decision(query: str, node: "TreeNode", options: List["TreeNode"],
                  memory: List[str], can_backtrack: bool) -> Dict:
    if options:
        opts_txt = "\n".join(f"[{i}] {o.name} — {_clip(o.summary, 380)}"
                             for i, o in enumerate(options))
    else:
        opts_txt = "(no remaining options here)"
    mem_txt = "\n".join(f"- {m}" for m in memory) or "(empty)"
    acts = ['"answer"']
    if options:
        acts.insert(0, '"descend"')
    if can_backtrack:
        acts.append('"backtrack"')
    prompt = (
        "You are an agent navigating a tree of document summaries to answer a "
        "question. You see only summaries (not the raw documents) and move one "
        "node at a time.\n\n"
        f"QUESTION: {query}\n\n"
        f"WORKING MEMORY (useful facts gathered so far):\n{mem_txt}\n\n"
        f"CURRENT NODE: {node.name} [{node.node_type}]\n"
        f"CURRENT SUMMARY: {_clip(node.summary, 900)}\n\n"
        f"CHILD OPTIONS:\n{opts_txt}\n\n"
        f"Choose ONE action ({', '.join(acts)}). Reply with ONLY a JSON object:\n"
        '{"reasoning": "<1-2 sentences>", '
        '"remember": "<a specific useful fact from the CURRENT summary worth keeping '
        'for the final answer, else empty>", '
        '"action": "descend|answer|backtrack", '
        '"child": <the [index] to descend into, or null>}\n'
        "Pick 'descend' with the index whose summary is most likely to lead to the "
        "answer. Pick 'answer' if working memory already answers the question or this "
        "is the most relevant place. Pick 'backtrack' if none of the options are "
        "relevant to the question."
    )
    return _parse_decision(_chat_text(prompt), len(options))


def memwalker(query: str, root: "TreeNode", strategy: str, verbose: bool = True) -> Dict:
    memory: List[str] = []
    evidence: List["TreeNode"] = []
    seen_evidence = set()
    visited = {root.node_id}
    trail: List[str] = []
    stack = [{"node": root, "options": get_nav_children(root, strategy), "tried": set()}]

    def log(msg):
        trail.append(msg)
        if verbose:
            print(msg)

    log(f"START at ROOT: {root.name}")
    steps = 0
    while stack and steps < MAX_STEPS:
        steps += 1
        fr = stack[-1]
        node = fr["node"]
        present = [(i, o) for i, o in enumerate(fr["options"])
                   if i not in fr["tried"] and o.node_id not in visited]
        opts = [o for _, o in present]
        dec = _ask_decision(query, node, opts, memory, can_backtrack=len(stack) > 1)
        if dec["remember"]:
            _add_memory(memory, dec["remember"])

        act = dec["action"]
        if act == "answer":
            log(f"  ANSWER here ({node.name}) — {dec['reasoning']}")
            if node.node_id not in seen_evidence:
                evidence.append(node); seen_evidence.add(node.node_id)
            break

        if act == "backtrack" or (act == "descend" and not opts):
            if len(stack) > 1:
                popped = stack.pop()
                parent = stack[-1]
                for i, o in enumerate(parent["options"]):
                    if o.node_id == popped["node"].node_id:
                        parent["tried"].add(i); break
                log(f"  BACKTRACK out of {popped['node'].name} — {dec['reasoning']}")
                continue
            log("  give up at root (nothing relevant) — answering from memory")
            break

        # descend
        ci = dec["child"]
        if ci is None or not (0 <= ci < len(opts)):
            ci = 0
        child = opts[ci]
        visited.add(child.node_id)
        child_opts = get_nav_children(child, strategy)
        if child.is_leaf() or not child_opts:
            log(f"  READ leaf: {child.name} ({child.metadata.get('source_file', '')})")
            if child.node_id not in seen_evidence:
                evidence.append(child); seen_evidence.add(child.node_id)
            if child.content:
                _add_memory(memory, _clip(child.content, 600))
            for i, o in enumerate(fr["options"]):       # don't offer it again here
                if o.node_id == child.node_id:
                    fr["tried"].add(i); break
            continue
        log(f"  DESCEND into {child.name} ({child.node_type}) — {dec['reasoning']}")
        stack.append({"node": child, "options": child_opts, "tried": set()})

    answer = _synthesize(query, memory, evidence)
    return {"answer": answer, "memory": memory, "evidence": evidence,
            "trail": trail, "steps": steps}


def _synthesize(query: str, memory: List[str], evidence: List["TreeNode"]) -> str:
    mem_txt = "\n".join(f"- {m}" for m in memory) or "(none)"
    ev_parts = []
    for e in evidence[:MAX_EVIDENCE]:
        src = e.metadata.get("source_file") or e.path or e.name
        ev_parts.append(f"[{src}]\n{_clip(e.content or e.summary, 1500)}")
    ev_txt = "\n\n".join(ev_parts) or "(none)"
    prompt = (
        "Answer the question using ONLY the information gathered below. If it is "
        "insufficient, say what is missing rather than guessing.\n\n"
        f"QUESTION: {query}\n\n"
        f"WORKING MEMORY:\n{mem_txt}\n\n"
        f"GATHERED EVIDENCE (most relevant nodes):\n{ev_txt}\n\n"
        "Write a clear, specific answer. Cite source files in [brackets] where relevant."
    )
    return _chat_text(prompt)


def answer_query(query: str, strategy: str = None, verbose: bool = True) -> Dict:
    strategy = strategy or VIRTUAL_STRATEGY
    print("=" * 80)
    print(f"QUERY: {query}")
    print(f"(strategy={strategy}, max_branch={MAX_BRANCH})")
    print("-" * 80)
    res = memwalker(query, ROOT, strategy, verbose=verbose)
    print("-" * 80)
    print(f"Visited {res['steps']} step(s); kept {len(res['memory'])} memory note(s), "
          f"{len(res['evidence'])} evidence node(s).")
    print("\nWORKING MEMORY:")
    for m in res["memory"]:
        print(f"  - {m}")
    print("\nANSWER:")
    print(res["answer"])
    print("=" * 80)
    return res


print("MemWalker agent ready (working memory + backtracking). Call answer_query('...').")

MemWalker agent ready (working memory + backtracking). Call answer_query('...').


**Back to benchmark code below.**

In [7]:
def _supports_think():
    # probe once, skip all thinking configs if the flag errors
    try:
        client.chat(model=AGENT_MODEL, messages=[{"role": "user", "content": "hi"}], options={"num_predict": 1}, think=True, keep_alive=KEEP_ALIVE)
        return True
    except TypeError:
        return False
    except Exception:
        return True


def _build_configs():
    # expand the grid, prune redundant or impossible combos
    think_ok = _supports_think()
    seen, order = {}, []
    for model, strat, think, gs, mb, cap in itertools.product(
            GRID["model"], GRID["strategy"], GRID["thinking"],
            GRID["group_summary"], GRID["max_branch"], GRID["decision_cap"]):
        if not any(model in a for a in AVAILABLE):
            continue                           # not pulled
        if think and not think_ok:
            continue
        eff_gs, eff_mb = gs, mb
        if strat == "none":                    # none ignores both of these
            eff_gs, eff_mb = "n/a", "n/a"
        key = (model, strat, think, eff_gs, eff_mb, cap)
        if key in seen:
            continue
        seen[key] = True
        order.append({"model": model, "strategy": strat, "thinking": think,
                      "group_summary": gs, "max_branch": mb, "decision_cap": cap})
    return order


def _apply(cfg):
    # push a config into the globals the agent reads, then wipe caches
    global AGENT_MODEL, VIRTUAL_STRATEGY, VIRTUAL_SUMMARY_LLM, MAX_BRANCH, DECISION_NUM_PREDICT
    AGENT_MODEL = cfg["model"]
    VIRTUAL_STRATEGY = cfg["strategy"]
    VIRTUAL_SUMMARY_LLM = (cfg["group_summary"] == "llm")
    MAX_BRANCH = cfg["max_branch"]
    DECISION_NUM_PREDICT = cfg["decision_cap"]
    _THINK["use"] = bool(cfg["thinking"])
    _nav_cache.clear()                         # stale virtual folders would cheat the timing
    _embed_cache.clear()


def _warm(models):
    # load each model once untimed so nobody pays the cold start
    for m in set(models):
        try:
            if m == EMBED_MODEL:
                client.embeddings(model=m, prompt="ok")    # embed models dont do chat
            else:
                client.chat(model=m, messages=[{"role": "user", "content": "ok"}], options={"num_predict": 1}, keep_alive=KEEP_ALIVE)
            print("warmed", m)
        except Exception as e:
            print("warm failed for", m, e)


CONFIGS = _build_configs()
print("configs to run:", len(CONFIGS), "x queries:", len(QUERIES), "x repeats:", REPEATS)
_warm([c["model"] for c in CONFIGS] + ([EMBED_MODEL] if any(c["strategy"] == "cluster" for c in CONFIGS) else []))

rows = []
responses = []                                 # full answers per config per query, for eyeballing quality
bar = tqdm(total=len(CONFIGS) * len(QUERIES) * REPEATS, desc="speed sweep", unit="run")   # eta comes from this
for ci, cfg in enumerate(CONFIGS, 1):
    _apply(cfg)
    _reset_stats()
    bar.set_postfix_str(f"{cfg['strategy']} think={cfg['thinking']} mb={cfg['max_branch']}")
    per_query, steps_acc, calls_acc = [], [], []
    ok, n = 0, 0
    t_cfg = time.perf_counter()
    cfg_ans = {}                               # this configs answer for each query
    for q in QUERIES:                          # same held queries every time
        for _ in range(REPEATS):
            c0 = _stats["decision_calls"] + _stats["synth_calls"] + _stats["group_calls"]
            t0 = time.perf_counter()
            res = memwalker(q, ROOT, VIRTUAL_STRATEGY, verbose=False)   # the real walk, quietly
            dt = time.perf_counter() - t0
            c1 = _stats["decision_calls"] + _stats["synth_calls"] + _stats["group_calls"]
            per_query.append(dt)
            steps_acc.append(res["steps"])
            calls_acc.append(c1 - c0)          # calls for just this query
            ok += 1 if res["answer"].strip() else 0
            n += 1
            bar.update(1)                      # one run done, tqdm updates the eta
        cfg_ans[q] = {"answer": res["answer"], "steps": res["steps"], "calls": c1 - c0}   # keep this querys answer
    total = time.perf_counter() - t_cfg
    rows.append({
        "strategy": cfg["strategy"], "thinking": cfg["thinking"],
        "group_summary": cfg["group_summary"] if cfg["strategy"] != "none" else "n/a",
        "max_branch": cfg["max_branch"] if cfg["strategy"] != "none" else "n/a",
        "decision_cap": cfg["decision_cap"], "model": cfg["model"],
        "mean_s": round(statistics.mean(per_query), 2),      # the headline number
        "mean_steps": round(statistics.mean(steps_acc), 1),
        "mean_calls": round(statistics.mean(calls_acc), 1),
        "chat_time_s": round(_stats["chat_time"], 1),
        "embed_time_s": round(_stats["embed_time"], 1),
        "ok_rate": round(ok / max(n, 1), 2),                 # did it answer at all
        "total_s": round(total, 1),
    })
    r = rows[-1]
    responses.append({"config": {"strategy": cfg["strategy"], "thinking": cfg["thinking"],
                                 "group_summary": r["group_summary"], "max_branch": r["max_branch"],
                                 "decision_cap": cfg["decision_cap"], "model": cfg["model"]},
                      "mean_s": r["mean_s"], "answers": cfg_ans})   # full answers for quality review
    tqdm.write(f"[{ci}/{len(CONFIGS)}] {cfg['strategy']:7} think={str(cfg['thinking']):5} "
               f"gs={str(r['group_summary']):9} mb={str(r['max_branch']):3} -> {r['mean_s']}s/query")
bar.close()
print("done; collected", len(rows), "rows")

configs to run: 26 x queries: 3 x repeats: 1
warmed nomic-embed-text
warmed gpt-oss:120b


speed sweep:   4%|█▋                                          | 3/78 [04:39<1:39:22, 79.50s/run, none think=True mb=5]

[1/26] none    think=False gs=n/a       mb=n/a -> 93.13s/query


speed sweep:   8%|███▏                                      | 6/78 [08:05<1:14:08, 61.79s/run, chunk think=False mb=5]

[2/26] none    think=True  gs=n/a       mb=n/a -> 68.59s/query


...

[24/26] cluster think=True  gs=llm       mb=5   -> 155.9s/query


speed sweep:  96%|████████████████████████████████████▌ | 75/78 [2:39:10<06:54, 138.09s/run, cluster think=True mb=12]

[25/26] cluster think=True  gs=llm       mb=8   -> 164.26s/query


speed sweep: 100%|██████████████████████████████████████| 78/78 [2:48:27<00:00, 129.59s/run, cluster think=True mb=12]

[26/26] cluster think=True  gs=llm       mb=12  -> 185.68s/query
done; collected 26 rows


In [8]:
df = pd.DataFrame(rows).sort_values("mean_s").reset_index(drop=True)   # fastest first
df.index = df.index + 1
print("\nLEADERBOARD, fastest first:")
print(df.to_string())

best = df.iloc[0]
print(f"\nfastest: strategy={best['strategy']} think={best['thinking']} "
      f"group_summary={best['group_summary']} max_branch={best['max_branch']} "
      f"-> {best['mean_s']}s per query")

df.to_csv(OUT_DIR / "speed_leaderboard.csv", index_label="rank")
df.to_json(OUT_DIR / "speed_leaderboard.json", orient="records", indent=2)
print("saved ->", OUT_DIR / "speed_leaderboard.csv")

resp_sorted = sorted(responses, key=lambda x: x["mean_s"])     # fastest first, matches the board
(OUT_DIR / "speed_responses.json").write_text(json.dumps(resp_sorted, indent=2), encoding="utf-8")
md = ["# full agent answers per config", ""]
for e in resp_sorted:
    c = e["config"]
    md.append(f"## {c['strategy']} | think={c['thinking']} | gs={c['group_summary']} | mb={c['max_branch']} | cap={c['decision_cap']} | {c['model']}")
    md.append(f"_mean {e['mean_s']}s per query_\n")
    for q, a in e["answers"].items():
        md.append(f"### Q: {q}")
        md.append(f"`steps={a['steps']} calls={a['calls']}`\n")
        md.append((a["answer"].strip() or "(empty)") + "\n")
(OUT_DIR / "speed_responses.md").write_text("\n".join(md), encoding="utf-8")
print("answers ->", OUT_DIR / "speed_responses.md")

try:                                           # chart is optional
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    tags = [f"{r.strategy[:3]}/{'T' if r.thinking else 'F'}/{str(r.group_summary)[:4]}/{r.max_branch}"
            for r in df.itertuples()]
    plt.figure(figsize=(10, max(3, len(df) * 0.35)))
    plt.barh(range(len(df)), df["mean_s"])
    plt.yticks(range(len(df)), tags, fontsize=7)
    plt.gca().invert_yaxis()                    # fastest on top to match the table
    plt.xlabel("mean seconds per query")
    plt.title("agent speed by config, lower is better")
    plt.tight_layout()
    plt.savefig(OUT_DIR / "speed_leaderboard.png", dpi=120)
    print("chart ->", OUT_DIR / "speed_leaderboard.png")
    plt.show()
except Exception as e:
    print("chart skipped:", e)


LEADERBOARD, fastest first:
   strategy  thinking group_summary max_branch  decision_cap         model  mean_s  mean_steps  mean_calls  chat_time_s  embed_time_s  ok_rate  total_s
1   cluster      True     heuristic          8           256  gpt-oss:120b   67.10        20.7        21.7        190.5          10.7      1.0    201.3
2      none      True           n/a        n/a           256  gpt-oss:120b   68.59        18.7        19.7        205.7           0.0      1.0    205.8
3   cluster     False     heuristic          8           256  gpt-oss:120b   70.05        20.7        21.7        200.1           9.9      1.0    210.2
4     chunk      True     heuristic          8           256  gpt-oss:120b   88.19        31.0        32.0        264.5           0.0      1.0    264.6
5     chunk     False     heuristic          8           256  gpt-oss:120b   88.63        31.0        32.0        265.8           0.0      1.0    265.9
6      none     False           n/a        n/a           25

Matplotlib is building the font cache; this may take a moment.


chart -> speedtest_out/speed_leaderboard.png


/var/folders/wr/cp6kpj5s3nx82dq7gl5vqrcr0000gp/T/ipykernel_52278/2437251721.py:44: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
